In [ ]:
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import cmocean as cmo
#from shapely.geometry import Polygon
from matplotlib.patches import Polygon
from copy import copy
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
import geopandas
import pickle

In [ ]:
lw_voronoi = 0.
lw_gl = .1
ec_voronoi = None #'w'

plotannual = True

x1,x2,y1,y2 = -1.75e6, -1.4e6,-0.75e6, -0.2e6
iwidth = .2

In [ ]:
#Allocate
cmaps = {}
norms = {}
ticks = {}
facs = {}
mvals = {}
labels = {}

#Create BMB colormap
vmax = 100
vmin = -10
linthresh = .3
linscale = .25
fracpos = (np.log10(vmax/linthresh)+linscale)/(np.log10(vmax/linthresh)+np.log10(-(vmin/linthresh))+2*linscale)
nneg = np.int_((1-fracpos)*256)
colors1 = plt.get_cmap('cmo.dense_r')(np.linspace(0,1.,nneg+1))
colors2 = plt.get_cmap('gist_heat_r')(np.linspace(0., 1, 256-nneg-1))
colors = np.vstack((colors1, colors2))

cmaps['BMB'] = mpl.colors.LinearSegmentedColormap.from_list('my_colormap', colors)
norms['BMB'] = mpl.colors.SymLogNorm(linthresh, vmin=vmin, vmax=vmax, linscale=linscale)
ticks['BMB'] = [-10,-3,-1,-.3,0,.3,1,3,10,30,100]
facs['BMB'] = -1
mvals['BMB'] = [4,5,6,8]
labels['BMB'] = 'Basal Mass Balance [m/yr]'

#Hi / Hs
#cmaps['Hi'] = plt.get_cmap('gist_stern')
cmaps['Hi'] = plt.get_cmap('cmo.ice')
norms['Hi'] = mpl.colors.Normalize(vmin=0,vmax=4000,clip=True)
ticks['Hi'] = np.arange(0,5000,1000)
facs['Hi'] = 1
mvals['Hi'] = [1,3,4,5,6,7,8,9,10]
labels['Hi'] = 'Ice thickness [m]'

#Ice speed
cmaps['uabs_surf'] = plt.get_cmap('CMRmap_r')
norms['uabs_surf'] = mpl.colors.LogNorm(vmin=1.,vmax=3000,clip=True)
ticks['uabs_surf'] = [1,10,100,1000]
facs['uabs_surf'] = 1
mvals['uabs_surf'] = [1,3,4,5,6,7,8,9,10]
labels['uabs_surf'] = 'Ice speed [m/yr]'

#Friction
cmaps['basal_friction_coefficient'] = plt.get_cmap('cmo.turbid')
norms['basal_friction_coefficient'] = mpl.colors.LogNorm(vmin=1e9,vmax=3e12,clip=True)
ticks['basal_friction_coefficient'] = [1e9,1e10,1e11,1e12]
facs['basal_friction_coefficient'] = 3600*24*365.25
mvals['basal_friction_coefficient'] = [1,3,7,9,10]
labels['basal_friction_coefficient'] = 'Basal friction [Pa m^-1 s]'

In [ ]:
gp_basins = geopandas.read_file("../../data/nsidc/0709/5000004499868/128409165/Basins_Antarctica_v02.shp")
gp_shelves = geopandas.read_file("../../data/nsidc/0709/5000004499868/128409162/IceShelf_Antarctica_v02.shp")


In [ ]:
run = 'ant_test_PIG_THW_CD_2'

pval0 = 'BMB'
pval1 = 'basal_friction_coefficient'
pval2 = 'uabs_surf'

fig = plt.figure(figsize=(10,6))

gs = fig.add_gridspec(2,6, height_ratios=(1, .05),width_ratios=(1,1,1,1,1,1),left=0.05, right=0.99, bottom=0.15, top=0.95, wspace=0.1, hspace=0.1)

ax0 = fig.add_subplot(gs[0,:3])
ax1 = fig.add_subplot(gs[0,3:])
cx0 = fig.add_subplot(gs[1,:2])
cx1 = fig.add_subplot(gs[1,2:4])
cx2 = fig.add_subplot(gs[1,4:])



#Add colorbars
for cax,pval in zip([cx0,cx1,cx2],[pval0,pval1,pval2]):

    cb = plt.colorbar(mpl.cm.ScalarMappable(norm=norms[pval], cmap=cmaps[pval]),cax=cax,extend='max',shrink=.8,orientation='horizontal')
    cb.set_label(labels[pval])
    cb.set_ticks(ticks[pval])
    #cb.set_ticklabels(ticks[pval])

#Gridded output
ds0 = xr.open_dataset(f'../output/{run}/main_output_ANT_grid.nc')
ds0 = ds0.isel(time=-1)

ax0.pcolormesh(ds0.x,ds0.y,facs[pval1]*ds0[pval1],cmap=cmaps[pval1],norm=norms[pval1])
ax0.pcolormesh(ds0.x,ds0.y,xr.where(ds0[pval0]!=0,facs[pval0]*ds0[pval0],np.nan),cmap=cmaps[pval0],norm=norms[pval0])
ax1.pcolormesh(ds0.x,ds0.y,facs[pval2]*ds0[pval2],cmap=cmaps[pval2],norm=norms[pval2])
ax0.set_aspect(1)
ax1.set_aspect(1)

#Add boundaries
for Ax in [ax0,ax1]:
    gp_basins.boundary.plot(ax=Ax,color='.5',linewidth=lw_gl)
    gp_shelves.boundary.plot(ax=Ax,color='.5',linewidth=lw_gl)

for Ax in [ax0,ax1]:
    Ax.set_xlim(-2.8e6, 2.8e6)
    Ax.set_ylim(-2.8e6, 2.8e6)
    Ax.set_xticks([])
    Ax.set_yticks([])
    Ax.set_title(f'UFE year {ds0.time.values:.0f}')

#Inset axes
ai0 = ax0.inset_axes([0.01, 0.01, iwidth, (y2-y1)/(x2-x1)*iwidth],xlim=(x1,x2),ylim=(y1,y2), xticklabels=[], yticklabels=[])
ai1 = ax1.inset_axes([0.01, 0.01, iwidth, (y2-y1)/(x2-x1)*iwidth],xlim=(x1,x2),ylim=(y1,y2), xticklabels=[], yticklabels=[])

ai0.pcolormesh(ds0.x,ds0.y,facs[pval1]*ds0[pval1],cmap=cmaps[pval1],norm=norms[pval1])
ai0.pcolormesh(ds0.x,ds0.y,xr.where(ds0[pval0]!=0,facs[pval0]*ds0[pval0],np.nan),cmap=cmaps[pval0],norm=norms[pval0])
ai1.pcolormesh(ds0.x,ds0.y,facs[pval2]*ds0[pval2],cmap=cmaps[pval2],norm=norms[pval2])
ai0.set_aspect(1)
ai1.set_aspect(1)

ax0.indicate_inset_zoom(ai0, edgecolor="black")
ax1.indicate_inset_zoom(ai1, edgecolor="black")

plt.savefig(f'../figures/gridzoom_{run}.png',dpi=600)

In [ ]:
t = 0
f = 0
run = 'spinup_hiv_u_notm'

pval0 = 'BMB'
pval1 = 'basal_friction_coefficient'
pval2 = 'uabs_surf'

for s in [1]:#range(1,25):

    #Open data on new mesh
    ds = xr.open_dataset(f'../output/{run}/main_output_ANT_{s:05d}.nc')
    ds = ds.isel(time=-1)

    try:
        dummy = plt.figure()
        new_manager = dummy.canvas.manager
        fig = pickle.load(open(f'../output/{run}/myfig_2p2_{s:05d}.pickle','rb'))
        new_manager.canvas.figure = fig
        fig.set_canvas(new_manager.canvas)
        fig = pickle.load(open(f'../output/{run}/myfig_2p2_{s:05d}.pickle','rb'))
        ax0 = fig.axes[0]
        ax1 = fig.axes[1]
        cx0 = fig.axes[2]
        cx1 = fig.axes[3]
        cx2 = fig.axes[4]
        ai0 = ax0.get_children()[-2]
        ai1 = ax1.get_children()[-2]

        print(f"{s:02d}: Loaded figure")
    except:

        #Create figure
        fig = plt.figure(figsize=(10,6))

        gs = fig.add_gridspec(2,6, height_ratios=(1, .05),width_ratios=(1,1,1,1,1,1),left=0.05, right=0.99, bottom=0.15, top=0.95, wspace=0.1, hspace=0.1)

        ax0 = fig.add_subplot(gs[0,:3])
        ax1 = fig.add_subplot(gs[0,3:])
        cx0 = fig.add_subplot(gs[1,:2])
        cx1 = fig.add_subplot(gs[1,2:4])
        cx2 = fig.add_subplot(gs[1,4:])
        ai0 = ax0.inset_axes([0.01, 0.01, iwidth, (y2-y1)/(x2-x1)*iwidth],xlim=(x1,x2),ylim=(y1,y2), xticklabels=[], yticklabels=[])
        ai1 = ax1.inset_axes([0.01, 0.01, iwidth, (y2-y1)/(x2-x1)*iwidth],xlim=(x1,x2),ylim=(y1,y2), xticklabels=[], yticklabels=[])
        
        
        #Make up axes and labels
        for Ax in [ax0,ax1,ai0,ai1]:
            Ax.set_aspect(1)
            #Ax.grid(color='.3',linewidth=.1)
            Ax.set_yticks([])
            Ax.set_xticks([])

        #Allocate for storage of polygons
        VAR0 = {}
        VAR1 = {}
        VAR2 = {}
        VAR3 = {}        
        
        # Gather points surrounding vertex (quasi-voronoi)
        
        for v in range(len(ds.vi)):
            polyx = []
            polyy = []
            for n in range(ds.niTri[v].data):
                c = ds.iTri[n,v].data-1 #Neighbouring triangle
                polyx.append(ds.Tricc[0,c].data)
                polyy.append(ds.Tricc[1,c].data)             
            #TO DO: Append edge points here

            #Accumulate into polygons
            points = np.array([polyx,polyy]).T
            VAR0[v] = Polygon(points,edgecolor=ec_voronoi,linewidth=lw_voronoi)
            VAR1[v] = Polygon(points,edgecolor=ec_voronoi,linewidth=lw_voronoi)
            VAR2[v] = Polygon(points,edgecolor=ec_voronoi,linewidth=lw_voronoi)
            VAR3[v] = Polygon(points,edgecolor=ec_voronoi,linewidth=lw_voronoi)

            #Add polygons to axis
            ax0.add_patch(VAR0[v])
            ax1.add_patch(VAR1[v])
            ai0.add_patch(VAR2[v])
            ai1.add_patch(VAR3[v])

        pickle.dump(fig, open(f'../output/{run}/myfig_2p2_{s:05d}.pickle', 'wb'))
        print(f"{s:02d}: Computed cells, saved axis")

    #Loop over vertex to fill colours and plot grounding line
    for v,(poly0,poly1,poly2,poly3) in enumerate(zip(ax0.patches,ax1.patches,ai0.patches,ai1.patches)):

        #Fill colours based on data
        if ds.mask[v].data in [2]: #Ocean
            poly0.set_facecolor('teal')
            poly1.set_facecolor('teal')
            poly2.set_facecolor('teal')
            poly3.set_facecolor('teal')            
        else:
            if ds.mask[v].data in mvals[pval0]:
                poly0.set_facecolor(cmaps[pval0](norms[pval0](facs[pval0]*ds[pval0][v])))
                poly2.set_facecolor(cmaps[pval0](norms[pval0](facs[pval0]*ds[pval0][v])))
            else:
                poly0.set_facecolor(cmaps[pval1](norms[pval1](facs[pval1]*ds[pval1][v])))
                poly2.set_facecolor(cmaps[pval1](norms[pval1](facs[pval1]*ds[pval1][v])))
            poly1.set_facecolor(cmaps[pval2](norms[pval2](facs[pval2]*ds[pval2][v])))
            poly3.set_facecolor(cmaps[pval2](norms[pval2](facs[pval2]*ds[pval2][v])))
            
        # Plot grounding line segments
        if ds.mask[v].data in [5,7]: #gl_gr or ca_gr
            for n in range(ds.nC[v].data):
                c = ds.C[n,v].data-1
                if ds.mask[c].data in [6,8]: #neighbouring gl_fl or ca_fl
                    cs = np.intersect1d(ds.iTri[:,v],ds.iTri[:,c])[1:]-1
                    ax0.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='yellow',zorder=10)
                    ai0.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='yellow',zorder=10)
                    ax1.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='tab:green',zorder=10)
                    ai1.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='tab:green',zorder=10)
                elif ds.mask[c].data in [2]: #neighbouring ocean: cliff
                    cs = np.intersect1d(ds.iTri[:,v],ds.iTri[:,c])[1:]-1
                    ax0.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='yellow',zorder=10)
                    ai0.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='yellow',zorder=10) 
                    ax1.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='tab:green',zorder=10)
                    ai1.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='tab:green',zorder=10)            
        
    ax0.set_title('Inversion')
    ax1.set_title(f"UFEMISM year {ds.time.values:10.0f}")

    mark_inset(ax0,ai0,loc1=2,loc2=4)
    mark_inset(ax1,ai1,loc1=2,loc2=4)

    #Add colorbars
    for cax,pval in zip([cx0,cx1,cx2],[pval0,pval1,pval2]):
        cb = plt.colorbar(mpl.cm.ScalarMappable(norm=norms[pval], cmap=cmaps[pval]),cax=cax,extend='max',shrink=.8,orientation='horizontal')
        cb.set_label(labels[pval])
        cb.set_ticks(ticks[pval])

    #Add boundaries
    for Ax in [ax0,ax1,ai0,ai1]:
        gp_basins.boundary.plot(ax=Ax,color='.5',linewidth=lw_gl,zorder=9)
        #gp_shelves.boundary.plot(ax=Ax,color='k',linewidth=2*lw_gl,zorder=9)

    for Ax in [ax0,ax1]:
        Ax.set_xlim(-2.8e6, 2.8e6)
        Ax.set_ylim(-2.8e6, 2.8e6)
    #Save figure
    canvas = FigureCanvas(fig)
    canvas.print_figure(f'../figures/2panels_{run}_{ds.time.values:.0f}.png',dpi=1800)

    #Zoom in and save
    #for Ax in [ax0,ax1]:
    #    Ax.set_xlim(-1.8e6, -0.8e6)
    #    Ax.set_ylim(-0.8e6, 0.2e6)
    #canvas = FigureCanvas(fig)
    #canvas.print_figure(f'../figures/2panels_{run}_zoom_{ds.time.values:.0f}.png',dpi=600)

    #Remove grounding line
    #for Ax in [ax1]:
    #    for line in Ax.get_lines():
    #        line.remove()

    print(f"{s:02d}: Made frame {f:03d} (time: {t:10.2f})")

    #Close data set to prepare for new mesh
    ds.close()

    plt.close()

    print(f"{s:02d}: All clear")

print(f"t = {t}")
print(f"f = {f}")

In [ ]:
t = 0
f = 0
run = 'spinup_hiv_u_tm_2'
pval = 'Hi'

ds1 = xr.open_dataset('../../data/ufemism/surface_velocity_measures_2km.nc')
ds2 = xr.open_dataset('../../data/BedMachineAntarctica-v3.nc')
mask = ds2.mask.values
mask = np.where(mask==3,0,mask)

for s in [1]:#range(1,25):

    #Open data on new mesh
    ds = xr.open_dataset(f'../output/{run}/main_output_ANT_{s:05d}.nc')
    ds = ds.isel(time=-1)

    try:
        dummy = plt.figure()
        new_manager = dummy.canvas.manager
        fig = pickle.load(open(f'../output/{run}/myfig_{s:05d}.pickle','rb'))
        new_manager.canvas.figure = fig
        fig.set_canvas(new_manager.canvas)
        fig = pickle.load(open(f'../output/{run}/myfig_{s:05d}.pickle','rb'))
        ax0 = fig.axes[0]
        ax1 = fig.axes[1]
        cax = fig.axes[2]
        print(f"{s:02d}: Loaded figure")
    except:

        #Create figure
        fig = plt.figure(figsize=(10,6))

        gs = fig.add_gridspec(2,2, height_ratios=(1, .05),width_ratios=(1,1),left=0.05, right=0.99, bottom=0.05, top=0.95, wspace=0.1, hspace=0.1)

        ax0 = fig.add_subplot(gs[0,0])
        ax1 = fig.add_subplot(gs[0,1])
        cax = fig.add_subplot(gs[1,:])

        #Make up axes and labels
        for Ax in [ax0,ax1]:
            Ax.set_aspect(1)
            #Ax.grid(color='.3',linewidth=.1)
            Ax.set_yticks([])
            Ax.set_xticks([])

        #Allocate for storage of polygons
        VAR = {}
        
        # Gather points surrounding vertex (quasi-voronoi)
        
        for v in range(len(ds.vi)):
            polyx = []
            polyy = []
            for n in range(ds.niTri[v].data):
                c = ds.iTri[n,v].data-1 #Neighbouring triangle
                polyx.append(ds.Tricc[0,c].data)
                polyy.append(ds.Tricc[1,c].data)             
            #TO DO: Append edge points here

            #Accumulate into polygons
            points = np.array([polyx,polyy]).T
            VAR[v] = Polygon(points,edgecolor=ec_voronoi,linewidth=lw_voronoi)

            #Add polygons to axis
            ax1.add_patch(VAR[v])

        pickle.dump(fig, open(f'../output/{run}/myfig_{s:05d}.pickle', 'wb'))
        print(f"{s:02d}: Computed cells, saved axis")

    #Loop over vertex to fill colours and plot grounding line
    for v,poly in enumerate(ax1.patches):

        #Fill colours based on data
        if ds.mask[v].data in [2]: #Ocean
            poly.set_facecolor('teal')
        elif ds.mask[v].data in mvals[pval]:
            poly.set_facecolor(cmaps[pval](norms[pval](facs[pval]*ds[pval][v])))
            
        # Plot grounding line segments
        if ds.mask[v].data in [5,7]: #gl_gr or ca_gr
            for n in range(ds.nC[v].data):
                c = ds.C[n,v].data-1
                if ds.mask[c].data in [6,8]: #neighbouring gl_fl or ca_fl
                    cs = np.intersect1d(ds.iTri[:,v],ds.iTri[:,c])[1:]-1
                    ax1.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='tab:blue',zorder=10)
                elif ds.mask[c].data in [2]: #neighbouring ocean: cliff
                    cs = np.intersect1d(ds.iTri[:,v],ds.iTri[:,c])[1:]-1
                    ax1.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='tab:blue',zorder=10)                       

    ax1.set_title(f"UFEMISM year {ds.time.values:10.0f}")

    #Add colorbars
    cb = plt.colorbar(mpl.cm.ScalarMappable(norm=norms[pval], cmap=cmaps[pval]),cax=cax,extend='max',shrink=.8,orientation='horizontal')

    cb.set_label(labels[pval])

    #Add observations
    ax0.pcolormesh(ds1.x,ds1.y,ds1.uabs_surf,cmap=cmaps['uabs_surf'],norm=norms['uabs_surf'])
    ax0.contour(ds2.x,ds2.y,mask,levels=[.5,100],colors='k',linewidths=lw_gl)
    ax0.set_title('Observations')

    #Add boundaries
    for Ax in [ax0,ax1]:
        gp_basins.boundary.plot(ax=Ax,color='.5',linewidth=lw_gl,zorder=9)
        #gp_shelves.boundary.plot(ax=Ax,color='k',linewidth=2*lw_gl,zorder=9)

    for Ax in [ax0,ax1]:
        Ax.set_xlim(-2.8e6, 2.8e6)
        Ax.set_ylim(-2.8e6, 2.8e6)
    #Save figure
    canvas = FigureCanvas(fig)
    canvas.print_figure(f'../figures/{pval}_{run}_{ds.time.values:.0f}.png',dpi=600)

    #Zoom in and save
    for Ax in [ax0,ax1]:
        Ax.set_xlim(-1.8e6, -0.8e6)
        Ax.set_ylim(-0.8e6, 0.2e6)
    canvas = FigureCanvas(fig)
    canvas.print_figure(f'../figures/{pval}_{run}_zoom_{ds.time.values:.0f}.png',dpi=600)

    #Remove grounding line
    for Ax in [ax1]:
        for line in Ax.get_lines():
            line.remove()

    print(f"{s:02d}: Made frame {f:03d} (time: {t:10.2f})")

    #Close data set to prepare for new mesh
    ds.close()

    plt.close()

    print(f"{s:02d}: All clear")

ds1.close()
ds2.close()

print(f"t = {t}")
print(f"f = {f}")

In [ ]:
#For Franka

run = 'spinup_hiv_u_tm_2'

pval0 = 'Hi' #shelf
pval1 = 'Hi' #grounded

s = 1
#Open data on new mesh
ds = xr.open_dataset(f'../output/{run}/main_output_ANT_{s:05d}.nc')
ds = ds.isel(time=-1)

try:
    #Load available pickle
    dummy = plt.figure()
    new_manager = dummy.canvas.manager
    fig = pickle.load(open(f'../output/{run}/myfig_1p_{s:05d}.pickle','rb'))
    new_manager.canvas.figure = fig
    fig.set_canvas(new_manager.canvas)
    fig = pickle.load(open(f'../output/{run}/myfig_1p_{s:05d}.pickle','rb'))
    ax0 = fig.axes[0]
    ax1 = fig.axes[1]
    cax = fig.axes[2]
    print(f"{s:02d}: Loaded pickle")
except:
    print(f"{s:02d}: No pickle available, making new one")
    #Create figure
    fig = plt.figure(figsize=(5,6))

    gs = fig.add_gridspec(2,2, height_ratios=(1, .05),width_ratios=(1,1),left=0.05, right=0.99, bottom=0.05, top=0.95, wspace=0.1, hspace=0.1)

    ax0 = fig.add_subplot(gs[0,:])
    cax0 = fig.add_subplot(gs[1,0])
    cax1 = fig.add_subplot(gs[1,1])

    #Make up axes and labels
    ax0.set_aspect(1)
    ax0.set_yticks([])
    ax0.set_xticks([])

    #Allocate for storage of polygons
    VAR = {}
    
    # Gather points surrounding vertex (quasi-voronoi)
    
    for v in range(len(ds.vi)):
        polyx = []
        polyy = []
        for n in range(ds.niTri[v].data):
            c = ds.iTri[n,v].data-1 #Neighbouring triangle
            polyx.append(ds.Tricc[0,c].data)
            polyy.append(ds.Tricc[1,c].data)             
        #TO DO: Append edge points here

        #Accumulate into polygons
        points = np.array([polyx,polyy]).T
        VAR[v] = Polygon(points,edgecolor=ec_voronoi,linewidth=lw_voronoi)

        #Add polygons to axis
        ax0.add_patch(VAR[v])

    pickle.dump(fig, open(f'../output/{run}/myfig_1p_{s:05d}.pickle', 'wb'))
    print(f"{s:02d}: Computed cells, saved pickle")

#Loop over vertex to fill colours and plot grounding line
for v,poly in enumerate(ax0.patches):

    #Fill colours based on data
    if ds.mask[v].data in [2]: #Ocean
        poly.set_alpha(0)
    elif ds.mask[v].data in mvals[pval0]: #Shelf
        poly.set_facecolor(cmaps[pval0](norms[pval0](facs[pval0]*ds[pval0][v])))
    elif ds.mask[v].data in mvals[pval1]: #Sheet
        poly.set_facecolor(cmaps[pval1](norms[pval1](facs[pval1]*ds[pval1][v])))
        
    # Plot grounding line segments
    if ds.mask[v].data in [5,7]: #gl_gr or ca_gr
        for n in range(ds.nC[v].data):
            c = ds.C[n,v].data-1
            if ds.mask[c].data in [6,8]: #neighbouring gl_fl or ca_fl
                cs = np.intersect1d(ds.iTri[:,v],ds.iTri[:,c])[1:]-1
                ax0.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='y',zorder=10)
            elif ds.mask[c].data in [2]: #neighbouring ocean: cliff
                cs = np.intersect1d(ds.iTri[:,v],ds.iTri[:,c])[1:]-1
                ax0.plot(ds.Tricc[0,cs],ds.Tricc[1,cs],lw=lw_gl,c='y',zorder=10)                       

#Add colorbars
cb = plt.colorbar(mpl.cm.ScalarMappable(norm=norms[pval0], cmap=cmaps[pval0]),cax=cax0,shrink=.8,orientation='horizontal')
cb.set_label(labels[pval0])

cb = plt.colorbar(mpl.cm.ScalarMappable(norm=norms[pval1], cmap=cmaps[pval1]),cax=cax1,shrink=.8,orientation='horizontal')
cb.set_label(labels[pval1])

#Add boundaries
gp_basins.boundary.plot(ax=ax0,color='.5',linewidth=lw_gl,zorder=9)

#Save figure
ax0.set_xlim(-2.8e6, 2.8e6)
ax0.set_ylim(-2.8e6, 2.8e6)
plt.savefig(f'../figures/{pval0}_{pval1}_{run}_{ds.time.values:.0f}.png',dpi=600,transparent=True)

#Zoom in and save
ax0.set_xlim(-1.8e6, -0.8e6)
ax0.set_ylim(-0.8e6, 0.2e6)
plt.savefig(f'../figures/{pval0}_{pval1}_{run}_zoom_{ds.time.values:.0f}.png',dpi=600,transparent=True)

#Close data set to prepare for new mesh
ds.close()



In [ ]:
#Make video
moviename = 'mismip_2km_05gl_WARM_FCMP'
if plotannual:
    framerate = 10
else:
    framerate = 40

command = f'ffmpeg -r {framerate} -f image2 -i ../video/frame_%03d.png -pix_fmt yuv420p -vcodec libx264 -crf 24 ../video/{moviename}.mp4'
print(command)
os.system(command)

#os.system('rm -r ../video/frame_*.png')